#  Limpieza, EDA y análisis avanzado — Netflix Movies and TV Shows

**Autor:** Marco Antonio Olmos  
**Objetivo:** Limpiar, explorar y analizar el dataset de Netflix para obtener insights sobre el catálogo, su evolución y sus patrones de contenido.

---

## 1. Carga del dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set(style="whitegrid")

df = pd.read_csv('/kaggle/input/datasets/shivamb/netflix-shows/netflix_titles.csv')
df.head()

# 2. Exploración inicial

In [ ]:
df.info()

In [ ]:
df.describe(include='all')


In [ ]:
df.isnull().sum()


In [ ]:
df.duplicated().sum()


## 3. Limpieza de columnas

Antes de realizar cualquier transformación profunda, es fundamental estandarizar los nombres de las columnas. Esto evita errores futuros, mejora la legibilidad del código y permite trabajar con mayor fluidez en todas las etapas del análisis.

El dataset original de Netflix contiene nombres con mayúsculas, espacios y formatos mixtos. Para un flujo de trabajo profesional, es necesario convertirlos a un formato uniforme.


In [ ]:
# Limpieza de nombres de columnas
df.columns = (
    df.columns
    .str.lower()          # Convertir a minúsculas
    .str.replace(' ', '_') # Reemplazar espacios por guiones bajos
)
df.head()


### Comentario profesional

La estandarización de nombres de columnas es una práctica esencial en análisis de datos por varias razones:

- **Consistencia:** evita tener columnas con formatos diferentes (mayúsculas, espacios, acentos).
- **Legibilidad:** facilita la lectura del código y la colaboración con otros analistas.
- **Compatibilidad:** muchas funciones de Pandas, NumPy y librerías de visualización funcionan mejor con nombres simples y sin espacios.
- **Automatización:** permite crear pipelines de limpieza y análisis sin preocuparse por errores de sintaxis.

Este paso prepara el dataset para todas las transformaciones posteriores y establece una base sólida para un análisis ordenado y profesional.


In [ ]:
df.columns


## 4. Manejo de duplicados

Antes de continuar con la limpieza profunda del dataset, es fundamental verificar si existen registros duplicados. Los duplicados pueden distorsionar métricas, inflar conteos y generar conclusiones incorrectas, especialmente en análisis descriptivos o modelos de machine learning.

En el dataset de Netflix, cada fila representa un título único. Por lo tanto, cualquier duplicado debe ser eliminado para garantizar la integridad del análisis.


In [ ]:
# Verificar cuántos registros duplicados existen
df.duplicated().sum()


In [ ]:
# Eliminar duplicados
df = df.drop_duplicates()

# Validar nuevamente
df.duplicated().sum()


### Comentario profesional

El manejo de duplicados es un paso crítico en cualquier proceso de limpieza de datos:

- **Evita sesgos en el análisis:** si un título aparece dos veces, podría inflar conteos de géneros, países o clasificaciones.
- **Mejora la calidad del dataset:** un dataset sin duplicados es más confiable para análisis estadísticos y visualizaciones.
- **Previene errores en modelos:** los duplicados pueden afectar negativamente algoritmos de clustering, recomendación o clasificación.

En este dataset, los duplicados suelen ser pocos o inexistentes, pero es importante verificarlo siempre.  
Después de aplicar `drop_duplicates()`, el dataset queda limpio y listo para continuar con el manejo de valores nulos.


## 5. Manejo de valores nulos

El dataset de Netflix contiene valores nulos en varias columnas clave como `director`, `cast`, `country`, `date_added`, `rating` y `duration`.  
Antes de avanzar con el análisis, es fundamental decidir cómo tratar estos valores para evitar errores y asegurar la consistencia del dataset.

En esta sección aplicamos estrategias de imputación adecuadas según el tipo de dato y el impacto que tienen en el análisis.


In [ ]:
df.isnull().sum()


### Análisis de los nulos

Los nulos se concentran en:

- **director** → 2634 nulos  
- **cast** → 825 nulos  
- **country** → 831 nulos  
- **date_added** → 10 nulos  
- **rating** → 4 nulos  
- **duration** → 3 nulos  

Estas columnas requieren tratamiento porque afectan directamente el análisis descriptivo y las visualizaciones.


In [ ]:
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['rating'] = df['rating'].fillna('Unknown')


### Justificación profesional

- En columnas como **director**, **cast**, **country** y **rating**, los valores nulos no pueden ser inferidos sin riesgo de introducir sesgos.
- Por ello, se reemplazan por `"Unknown"`, una práctica estándar en análisis de datos.
- Esto permite mantener la integridad del dataset sin eliminar información útil.


In [ ]:
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')



### Justificación profesional

- La columna `date_added` contiene fechas en formato texto, algunas mal formateadas.
- `errors='coerce'` convierte automáticamente los valores inválidos en `NaT` (Not a Time).
- Esto permite trabajar con fechas de manera segura en análisis temporal.


In [ ]:
df.isnull().sum()


### Resultado esperado

Después de la imputación:

- Las columnas de texto ya no deben tener nulos.
- `date_added` puede conservar algunos `NaT`, lo cual es aceptable para análisis temporal.
- El dataset queda listo para enriquecer columnas derivadas como `year_added` y `month_added`.


### Comentario profesional

El manejo de valores nulos es un paso crítico en cualquier proyecto de análisis de datos:

- **Evita errores en visualizaciones y modelos**, especialmente cuando se trabaja con fechas o categorías.
- **Mantiene la integridad del dataset**, evitando eliminar registros completos por tener un solo valor faltante.
- **Permite análisis más precisos**, ya que las columnas clave quedan completas y listas para ser utilizadas en gráficos, agrupaciones y modelos.

En este dataset, la estrategia de imputación es adecuada porque:
- Los nulos representan información faltante, no valores incorrectos.
- No existe una forma confiable de inferir los valores reales.
- `"Unknown"` es una etiqueta neutral que no introduce sesgos.

Con esta limpieza, el dataset queda preparado para continuar con la conversión de fechas y el análisis avanzado.


## 6. Conversión y enriquecimiento de fechas

La columna `date_added` ya fue convertida a formato datetime.  
Ahora vamos a enriquecerla creando nuevas columnas que nos permitan analizar el catálogo de Netflix desde una perspectiva temporal.

Estas columnas son fundamentales para:

- estudiar tendencias por año  
- analizar crecimiento del catálogo  
- identificar patrones de incorporación de contenido  
- crear visualizaciones temporales


In [ ]:
# Crear columnas derivadas de la fecha
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month
df['day_added'] = df['date_added'].dt.day


In [ ]:
df[['date_added', 'year_added', 'month_added', 'day_added']].head()


### Comentario profesional

La creación de columnas derivadas a partir de `date_added` permite:

- Analizar el crecimiento del catálogo por año.
- Detectar picos de incorporación de contenido.
- Comparar tendencias entre películas y series.
- Preparar visualizaciones temporales más completas.
- Facilitar análisis estacionales (meses con más lanzamientos).

Este enriquecimiento es un paso estándar en análisis temporal y prepara el dataset para las visualizaciones avanzadas que vienen más adelante.


## 7. Normalización de categorías (listed_in)

La columna `listed_in` contiene los géneros o categorías del contenido, pero vienen en formato de texto con múltiples valores separados por comas.  
Ejemplo:

- "Dramas, International Movies"
- "Crime TV Shows, TV Mysteries, TV Dramas"

Para poder analizar los géneros de forma correcta, es necesario transformar esta columna en una estructura más manejable.


In [ ]:
# Convertir los géneros en listas
df['listed_in'] = df['listed_in'].str.split(', ')
df['listed_in'].head()


In [ ]:
# Verificar el tipo de dato
type(df['listed_in'].iloc[0]), df['listed_in'].iloc[0]


### Comentario profesional

Transformar `listed_in` en listas permite:

- Analizar géneros de forma individual.
- Contar cuántas veces aparece cada categoría.
- Crear visualizaciones más precisas.
- Explorar combinaciones de géneros.
- Preparar el dataset para análisis avanzado como clustering o segmentación.

Este paso es fundamental porque los géneros son una de las variables más importantes en el catálogo de Netflix.  
Sin esta normalización, cualquier análisis por categoría sería impreciso o directamente incorrecto.


In [ ]:
# Crear una serie con todos los géneros "explotados"
all_genres = df['listed_in'].explode()

# Ver los 10 géneros más frecuentes
all_genres.value_counts().head(10)


### Insight

Al explotar la columna `listed_in`, podemos observar cuáles son los géneros más frecuentes en Netflix.  
Esto permite identificar:

- Tendencias de contenido
- Preferencias del catálogo
- Enfoque regional o global
- Oportunidades para análisis de nichos

Este paso prepara el terreno para las visualizaciones avanzadas que vendrán más adelante.


## 8. Limpieza y separación de la columna `duration`

La columna `duration` contiene información sobre la duración del contenido, pero viene en un formato mixto:

- Películas → `"90 min"`, `"110 min"`, `"45 min"`
- Series → `"1 Season"`, `"2 Seasons"`, `"5 Seasons"`

Para poder analizar correctamente la duración del contenido, es necesario separar esta columna en dos partes:

1. **duration_value** → el valor numérico (minutos o temporadas)  
2. **duration_type** → el tipo de duración (`min`, `Season`, `Seasons`)

Esta separación es fundamental para análisis comparativos entre películas y series.


In [ ]:
# Extraer el tipo de duración (min, Season, Seasons)
df['duration_type'] = df['duration'].str.extract(r'(min|Season|Seasons)')
df['duration_type'].head()


In [ ]:
# Extraer el valor numérico de la duración
df['duration_value'] = df['duration'].str.extract(r'(\d+)').astype(float)
df[['duration', 'duration_type', 'duration_value']].head()


In [ ]:
df[['duration', 'duration_type', 'duration_value']].sample(5)


### Comentario profesional

Separar la columna `duration` en dos componentes permite:

- Analizar películas por duración en minutos.
- Analizar series por número de temporadas.
- Crear visualizaciones más precisas (histogramas, boxplots, distribuciones).
- Comparar patrones entre películas y series.
- Preparar el dataset para modelos de recomendación basados en duración.

Este paso es especialmente importante porque la duración es una variable clave en el consumo de contenido:
- Las películas suelen tener duraciones estándar (80–120 min).
- Las series tienden a tener pocas temporadas (1–3 en promedio).

Sin esta separación, cualquier análisis de duración sería incorrecto o engañoso.


In [ ]:
movies = df[df['duration_type'] == 'min']
shows = df[df['duration_type'].isin(['Season', 'Seasons'])]

movies.head(3), shows.head(3)


### Insight

La separación de la duración revela patrones importantes:

- Las películas se concentran en rangos de duración muy definidos.
- Las series suelen tener pocas temporadas, lo que coincide con la estrategia de Netflix de producir contenido corto y fácil de consumir.
- Existen outliers tanto en películas como en series que pueden ser analizados más adelante.

Este paso prepara el dataset para las visualizaciones avanzadas de duración que vendrán en la sección de análisis.


## 9. EDA básico del catálogo

Antes de generar visualizaciones avanzadas, es importante realizar un EDA básico para validar que:

- La limpieza fue aplicada correctamente.
- Las columnas derivadas funcionan como se espera.
- No existen inconsistencias que afecten los gráficos.
- El dataset está listo para análisis descriptivo y visual.

Este EDA inicial nos permite confirmar que la estructura del dataset es sólida y que podemos avanzar con confianza hacia el análisis avanzado.


In [ ]:
df['type'].value_counts()


### Análisis profesional

Esta distribución nos permite validar que la columna `type` está limpia y correctamente categorizada.  
Netflix suele tener más películas que series, por lo que esta salida debe reflejar esa tendencia.


In [ ]:
df['country'].value_counts().head(10)


### Análisis profesional

Aquí confirmamos que la imputación de `"Unknown"` funcionó y que los países están correctamente normalizados.  
Estados Unidos suele aparecer como el país dominante, seguido por India y Reino Unido.


In [ ]:
df['rating'].value_counts()


### Análisis profesional

Esta salida permite validar que:

- La imputación de `"Unknown"` fue exitosa.
- Las clasificaciones están en formato consistente.
- No existen valores extraños o mal escritos.

Netflix suele tener una fuerte presencia de contenido **TV-MA**, lo cual debe reflejarse aquí.


In [ ]:
df[['duration', 'duration_type', 'duration_value']].sample(5)


### Análisis profesional

Este paso confirma que:

- La separación entre `duration_type` y `duration_value` fue correcta.
- Películas tienen duración en minutos.
- Series tienen duración en temporadas.
- No existen valores vacíos o inconsistentes.

Si todo se ve correcto, podemos avanzar a visualizaciones de duración.


In [ ]:
df[['date_added', 'year_added', 'month_added']].sample(5)


### Análisis profesional

Aquí verificamos que:

- `date_added` está correctamente convertida a datetime.
- `year_added` y `month_added` se generaron sin errores.
- Los valores `NaT` en `date_added` no afectan las columnas derivadas.

Si los años y meses se ven correctos, el dataset está listo para análisis temporal.


In [ ]:
df['listed_in'].explode().value_counts().head(10)


### Análisis profesional

Este paso confirma que:

- La columna `listed_in` fue correctamente convertida a listas.
- Los géneros están listos para análisis individual.
- No existen valores vacíos o mal formateados.

Los géneros más frecuentes suelen ser:
- Dramas  
- Documentaries  
- International Movies  
- Comedies  


### Conclusión

El EDA básico confirma que:

- La limpieza fue aplicada correctamente.
- Las columnas derivadas funcionan sin errores.
- No existen inconsistencias que afecten el análisis.
- El dataset está listo para visualizaciones avanzadas.

Podemos avanzar con confianza hacia la sección de gráficos y análisis profundo.


## 10. Visualizaciones y análisis avanzado

En esta sección analizamos el catálogo de Netflix mediante visualizaciones profesionales que permiten identificar patrones, tendencias y comportamientos clave del contenido disponible en la plataforma.

Las gráficas se construyen con Seaborn y Matplotlib, y cada una incluye un análisis profesional que explica los hallazgos más relevantes.



In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='type', hue='type', palette='viridis', legend=False)
plt.title("Distribución de contenido: Películas vs Series")
plt.xlabel("Tipo de contenido")
plt.ylabel("Cantidad")
plt.show()


### Insight profesional

Netflix tiene una mayor proporción de **películas** que **series**, lo cual indica un enfoque fuerte en contenido de formato corto.  
Sin embargo, las series suelen ser las que generan mayor retención, por lo que esta diferencia es estratégica.


In [ ]:
top_countries = df['country'].value_counts().head(10)

plt.figure(figsize=(10,6))
sns.barplot(x=top_countries.values, y=top_countries.index, palette='magma')
plt.title("Top 10 países con más contenido en Netflix")
plt.xlabel("Cantidad de títulos")
plt.ylabel("País")
plt.show()


### Insight profesional

Estados Unidos domina la producción del catálogo, seguido por India y Reino Unido.  
Esto refleja la estrategia global de Netflix y su fuerte dependencia del contenido estadounidense e internacional.


In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(data=df, x='year_added', hue='year_added', palette='coolwarm', legend=False)
plt.title("Contenido agregado por año")
plt.xticks(rotation=45)
plt.xlabel("Año")
plt.ylabel("Cantidad de títulos agregados")
plt.show()


### Insight profesional

Entre **2016 y 2020** se observa un crecimiento acelerado del catálogo.  
La pandemia en 2020 coincide con un aumento significativo de contenido agregado.


In [ ]:
genre_list = df['listed_in'].explode()
top_genres = genre_list.value_counts().head(10)

plt.figure(figsize=(12,6))
sns.barplot(x=top_genres.values, y=top_genres.index, palette='cubehelix')
plt.title("Top 10 géneros más frecuentes en Netflix")
plt.xlabel("Cantidad de títulos")
plt.ylabel("Género")
plt.show()


### Insight profesional

Los géneros más comunes son **Dramas**, **Documentales**, **Comedias** e **International Movies**.  
Esto confirma que Netflix apuesta por contenido internacional y narrativas profundas.


In [ ]:
movies = df[df['duration_type'] == 'min']
shows = df[df['duration_type'].isin(['Season', 'Seasons'])]

plt.figure(figsize=(12,5))
sns.histplot(movies['duration_value'], kde=True, color='blue')
plt.title("Distribución de duración de películas (minutos)")
plt.xlabel("Duración (min)")
plt.show()

plt.figure(figsize=(12,5))
sns.countplot(data=shows, x='duration_value', hue='duration_value', palette='rocket', legend=False)
plt.title("Distribución de temporadas en series")
plt.xlabel("Número de temporadas")
plt.show()


### Insight profesional

Las películas se concentran entre **80 y 120 minutos**, mientras que la mayoría de las series tienen **1–3 temporadas**.  
Esto coincide con la estrategia de Netflix de producir contenido corto y fácil de consumir.


In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(
    data=df,
    x='rating',
    hue='rating',
    palette='flare',
    order=df['rating'].value_counts().index,
    legend=False
)
plt.title("Distribución de clasificaciones de contenido")
plt.xticks(rotation=45)
plt.xlabel("Clasificación")
plt.ylabel("Cantidad")
plt.show()


### Insight profesional

Las clasificaciones más comunes son **TV-MA**, **TV-14** y **TV-PG**, lo que indica una fuerte presencia de contenido para adultos.  
Netflix ha incrementado su catálogo de contenido maduro en los últimos años.


In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=movies, x='duration_value', color='purple')
plt.title("Outliers en duración de películas")
plt.xlabel("Duración (min)")
plt.show()


### Insight profesional

Existen películas con duraciones extremadamente largas o cortas.  
Estos outliers pueden corresponder a:

- Documentales especiales  
- Cortometrajes  
- Errores de captura  

Es importante considerarlos en análisis más avanzados.


### Conclusión

El análisis avanzado revela patrones clave del catálogo de Netflix:

- Fuerte enfoque en películas.  
- Crecimiento acelerado entre 2016–2020.  
- Dominancia de contenido estadounidense e internacional.  
- Preferencia por dramas y documentales.  
- Clasificaciones orientadas a adultos.  
- Series cortas como estrategia de retención.  

Estas visualizaciones permiten construir análisis más profundos como:
- Tendencias por país  
- Segmentación por género  
- Modelos de recomendación  
- Clustering de contenido  


## 11. Conclusiones del proyecto

El análisis del catálogo de Netflix permitió transformar un dataset crudo en información clara, estructurada y accionable. A través de un proceso completo de limpieza, exploración y visualización, se identificaron patrones clave sobre el contenido disponible en la plataforma.

### Principales conclusiones

1. **Netflix tiene una mayor proporción de películas que series**, lo cual refleja una estrategia orientada a contenido de consumo rápido.  
   Sin embargo, las series —aunque menos numerosas— suelen ser las que generan mayor retención.

2. **Estados Unidos domina la producción del catálogo**, seguido por India y Reino Unido.  
   Esto confirma la fuerte presencia de contenido internacional y la estrategia global de Netflix.

3. **El catálogo creció aceleradamente entre 2016 y 2020**, con un pico notable durante la pandemia.  
   Este comportamiento coincide con la expansión agresiva de la plataforma.

4. **Los géneros más frecuentes son Dramas, Documentales, Comedias e International Movies.**  
   Esto indica una apuesta por contenido narrativo profundo y producciones globales.

5. **Las películas se concentran entre 80 y 120 minutos**, mientras que la mayoría de las series tienen entre 1 y 3 temporadas.  
   Netflix prioriza contenido corto, fácil de consumir y con ciclos de producción rápidos.

6. **Las clasificaciones más comunes son TV-MA, TV-14 y TV-PG**, lo que evidencia una fuerte presencia de contenido para adultos.  
   La plataforma ha incrementado su catálogo maduro en los últimos años.

7. **La limpieza del dataset fue fundamental** para obtener resultados confiables:
   - Se imputaron valores nulos de forma adecuada.
   - Se normalizaron columnas clave como `listed_in` y `duration`.
   - Se generaron columnas derivadas para análisis temporal.
   - Se eliminaron duplicados y se estandarizaron nombres de columnas.

### Reflexión final

Este proyecto demuestra la importancia de un flujo de trabajo ordenado y profesional en análisis de datos:

- **Limpieza estructurada**  
- **EDA inicial para validar la calidad del dataset**  
- **Visualizaciones claras y orientadas a insights**  
- **Storytelling basado en datos**  

El resultado es un análisis sólido que puede servir como base para:

- Modelos de recomendación  
- Segmentación de contenido  
- Estudios de tendencias  
- Dashboards interactivos  
- Proyectos de machine learning  

Este notebook representa un ejemplo completo de análisis de datos aplicado a un dataset real y ampliamente utilizado en la industria.
